# Modeling and Results

This notebook contains the current reproducible baseline model. The next pass should add tuned Random Forest and boosting models here.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_merged_data, summarize_data_quality
from src.plots import set_plot_style

set_plot_style()

from sklearn.metrics import classification_report

from src.config import BASELINE_FEATURES
from src.feature_engineering import add_next_close_target, select_baseline_xy
from src.modeling import chronological_train_test_split, scale_train_test, fit_baseline_logistic_regression
from src.evaluation import evaluate_classifier
from src.plots import plot_confusion_matrix

df = load_merged_data()
baseline_df = add_next_close_target(df)


## Baseline Train/Test Split

The split is chronological (`shuffle=False`) to avoid leaking future market behavior into training.

In [ ]:
X, y = select_baseline_xy(baseline_df, BASELINE_FEATURES)
X_train, X_test, y_train, y_test = chronological_train_test_split(X, y)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")


## Baseline Logistic Regression

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_train_test(X_train, X_test)
log_reg = fit_baseline_logistic_regression(X_train_scaled, y_train)
log_reg


## Baseline Evaluation

In [ ]:
results = evaluate_classifier(log_reg, X_test_scaled, y_test, label="Baseline Logistic Regression")

print("--- Baseline Logistic Regression Results ---\n")
print(classification_report(y_test, results["predictions"]))
print("ROC-AUC Score:", results["roc_auc"])

display(results["classification_report_df"])

fig = plot_confusion_matrix(results["confusion_matrix"])
plt.show()


## Next Modeling Work

- Replace the post-level target with the final cleaned hourly/hybrid target.
- Add at least two stronger models: Random Forest and boosting.
- Add RandomizedSearchCV or Bayesian/random search tuning.
- Report precision, recall, F1, ROC-AUC, and preferably PR-AUC.
- Add feature importance from the best tree-based model.